# RAG Pipeline — PDF Analysis with LangChain
## End-to-end workflow: ingest PDFs → build vector store → run analysis queries → export CSV + HTML report with charts

### Architecture Overview
```
PDF Files  →  Text Extraction  →  Chunking  →  ChromaDB (Vector Store)
                                                        ↓
                                          RAG Chain (LLM + Retriever + Reranker)
                                                        ↓
                               Structured Analysis (topics, entities, sentiment)
                                                        ↓
                             RAG Evaluation (context recall, answer faithfulness)
                                                        ↓
                                  CSV Export  +  Plotly Charts  →  HTML Report
```

### Steps
1. Environment Setup & Package Installation
2. Configuration (paths, API keys, model settings)
3. PDF Ingestion & Text Extraction
4. Text Chunking & Preprocessing
5. Embeddings & ChromaDB Vector Store
6. RAG Chain Construction (MMR + Cohere Reranker)
7. Batch Document Analysis
8. Results Export to CSV
9. Visualizations with Plotly
10. RAG Evaluation & Quality Metrics
11. HTML Report Generation


## Step 1 — Environment Setup & Package Installation

In [1]:
# 📦 Install all dependencies (run once)
!pip install -q -r requirements.txt

## Step 2 — Configuration

Set your PDF folder path, API keys, model choices, and chunking parameters here.
All subsequent cells read from this configuration block.

In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads .env file if present

# ── Paths ──────────────────────────────────────────────────────────────────
PDF_DIR         = Path("./pdfs")            # folder containing your PDF files
VECTORSTORE_DIR = Path("./vectorstore")     # ChromaDB persistence directory
OUTPUT_DIR      = Path("./output")          # CSV + HTML output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_OUTPUT      = OUTPUT_DIR / "analysis_results.csv"
HTML_REPORT     = OUTPUT_DIR / "report.html"

# ── LLM Provider ───────────────────────────────────────────────────────────
# Options: "openai" | "cohere"
LLM_PROVIDER    = "openai"
OPENAI_API_KEY  = os.getenv("OPENAI_API_KEY", "sk-...")
COHERE_API_KEY  = os.getenv("COHERE_API_KEY", "")

# ── Model names ────────────────────────────────────────────────────────────
OPENAI_MODEL    = "gpt-4o-mini"             # or "gpt-3.5-turbo"
COHERE_MODEL    = "command-r"
EMBEDDING_MODEL = "all-mpnet-base-v2"       # HuggingFace sentence-transformer

# ── Chunking ────────────────────────────────────────────────────────────────
CHUNK_SIZE      = 1024
CHUNK_OVERLAP   = 128
TOP_K_RETRIEVAL = 4                          # docs returned per query

# ── Analysis queries — Software Solution Architecture learning materials ───
ANALYSIS_QUERIES = {
    # ── Overview ────────────────────────────────────────────────────────────
    "summary": (
        "Provide a concise 3-sentence summary of this document, "
        "focusing on its purpose within a Software Solution Architecture curriculum."
    ),
    "learning_objectives": (
        "List the key learning objectives or competencies a student should gain "
        "from this material. Return as a numbered list."
    ),
    "module_topic": (
        "What is the primary architectural topic or domain covered by this document? "
        "You MUST pick exactly ONE label from this fixed list — do not invent new labels:\n"
        "  Solution Architecture, Microservices, Event-Driven Architecture, "
        "Cloud-Native Design, API Design, Security Architecture, "
        "Data Architecture, DevOps & CI/CD, System Design, "
        "Enterprise Architecture, Software Engineering Fundamentals, "
        "Governance & Standards\n"
        "Return only the label, nothing else."
    ),

    # ── Architecture content ────────────────────────────────────────────────
    "architecture_patterns": (
        "List all software architecture patterns, styles, or principles explicitly "
        "mentioned or taught (e.g. CQRS, Saga, Strangler Fig, Hexagonal, "
        "Event Sourcing, 12-Factor). Return as comma-separated values."
    ),
    "technologies_tools": (
        "List all specific technologies, frameworks, cloud services, or tools "
        "referenced (e.g. Kafka, Kubernetes, AWS Lambda, OAuth2, OpenAPI). "
        "Return as comma-separated values."
    ),
    "architectural_decisions": (
        "What key architectural trade-offs, decision criteria, or ADR (Architecture "
        "Decision Record) topics are discussed? Summarise in 2–3 sentences."
    ),

    # ── Pedagogy ────────────────────────────────────────────────────────────
    "difficulty_level": (
        "Assess the difficulty level of this material for a software architect student. "
        "Answer with one word: Introductory, Intermediate, or Advanced."
    ),
    "prerequisites": (
        "What prior knowledge or skills does this material assume the student already has? "
        "List as comma-separated values, or 'None stated' if not mentioned."
    ),
    "exercises_labs": (
        "Are there any practical exercises, labs, case studies, or assignments described? "
        "If yes, briefly describe them. If no, answer 'None'."
    ),

    # ── Quality & completeness ──────────────────────────────────────────────
    "key_concepts": (
        "List the top 7 key architectural concepts or terms defined or explained "
        "in this document. Return as comma-separated values."
    ),
    "gaps_recommendations": (
        "Identify any apparent gaps, outdated content, or areas that could be "
        "strengthened in this learning material from an enterprise architecture perspective."
    ),
}

print("✅ Configuration loaded")
print(f"   PDF source  : {PDF_DIR.resolve()}")
print(f"   Vector store: {VECTORSTORE_DIR.resolve()}")
print(f"   Analysis queries: {len(ANALYSIS_QUERIES)} defined")
print(f"   Output dir  : {OUTPUT_DIR.resolve()}")
print(f"   LLM         : {LLM_PROVIDER} / {OPENAI_MODEL if LLM_PROVIDER == 'openai' else COHERE_MODEL}")


✅ Configuration loaded
   PDF source  : /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/pdfs
   Vector store: /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/vectorstore
   Analysis queries: 11 defined
   Output dir  : /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/output
   LLM         : openai / gpt-4o-mini


## Step 3 — PDF Ingestion & Text Extraction

`PyPDFLoader` reads each PDF page-by-page and attaches metadata:
- `source` — file path
- `page` — page number (0-indexed)
- `filename` — base filename (used later as document identity)

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.schema import Document
from typing import List
import glob

def load_pdfs(pdf_dir: Path) -> List[Document]:
    """Load all PDFs from a directory.  Returns a flat list of page-level Documents."""
    pdf_files = sorted(pdf_dir.glob("*.pdf"))
    
    if not pdf_files:
        raise FileNotFoundError(
            f"No PDF files found in '{pdf_dir.resolve()}'.\n"
            "Please copy your PDF files into that folder and re-run this cell."
        )
    
    all_docs: List[Document] = []
    print(f"Found {len(pdf_files)} PDF file(s) in {pdf_dir.resolve()}\n")
    
    for pdf_path in pdf_files:
        loader = PyPDFLoader(str(pdf_path))
        pages  = loader.load()
        
        # Enrich every page with a clean filename tag
        for page in pages:
            page.metadata["filename"] = pdf_path.name
            page.metadata["filepath"] = str(pdf_path)
        
        all_docs.extend(pages)
        print(f"  [{pdf_path.name}]  → {len(pages)} page(s) loaded")
    
    print(f"\nTotal pages extracted: {len(all_docs)}")
    return all_docs

# Create the PDF folder if it doesn't exist yet
PDF_DIR.mkdir(parents=True, exist_ok=True)

raw_docs = load_pdfs(PDF_DIR)

Found 20 PDF file(s) in /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/pdfs

  [SAS_-_Module_1_-_Program___SA_Introduction__eLearning__v4.3.pdf]  → 86 page(s) loaded
  [SAS_-_Module_2_-_Business_Architecture_v4.3.pdf]  → 63 page(s) loaded
  [SAS_-_Module_3.1_-_ASRs_v4.1.pdf]  → 61 page(s) loaded
  [SAS_-_Module_3.2_-_Quality_Attributes_v4.4.pdf]  → 63 page(s) loaded
  [SAS_-_Module_4_-_Architectural_Styles_and_Patterns_v4.1.pdf]  → 104 page(s) loaded
  [SAS_-_Module_5_-_Architecture_Modeling_v4.pdf]  → 74 page(s) loaded
  [SAS_-_Module_6_-_Architecture_Documentation_v4.pdf]  → 119 page(s) loaded
  [SAS_-_Module_7.1_-_Pre-sales_v4.pdf]  → 77 page(s) loaded
  [SAS_-_Module_7.2_-_Estimation_v4.0.pdf]  → 55 page(s) loaded
  [SAS_-_Module_7.3_-_Discovery__Construction__Transition_v4.1.pdf]  → 62 page(s) loaded
  [SAS_-_Module_8.1_-_Architecture_Design_v4.pdf]  → 43 page(s) loaded
  [SAS_-_Module_8.2_-_Architecture_Review_v4.pdf]  → 42 page(s) loaded
  [SAS_-_Module_9.1.1_-_Technology_Domains_-_C

## Step 4 — Text Chunking & Preprocessing

Split the page-level documents into smaller, overlapping chunks suitable for embedding.
`RecursiveCharacterTextSplitter` tries to break on paragraphs → sentences → words in order,
preserving semantic coherence within each chunk.

In [4]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""],
    length_function=len,
)

chunked_docs: List[Document] = splitter.split_documents(raw_docs)

# Add a chunk index to each document for traceability
for idx, doc in enumerate(chunked_docs):
    doc.metadata["chunk_id"] = idx

# Quick stats
import pandas as pd

stats_df = (
    pd.DataFrame([
        {
            "filename": d.metadata.get("filename", "unknown"),
            "page":     d.metadata.get("page", -1),
            "chunk_id": d.metadata.get("chunk_id"),
            "chars":    len(d.page_content),
        }
        for d in chunked_docs
    ])
)

print(f"Total chunks : {len(chunked_docs)}")
print(f"Avg chunk size: {stats_df['chars'].mean():.0f} chars\n")
print("Chunks per document:")
print(stats_df.groupby("filename")["chunk_id"].count().to_string())

Total chunks : 1681
Avg chunk size: 461 chars

Chunks per document:
filename
SAS_-_Module_1_-_Program___SA_Introduction__eLearning__v4.3.pdf                               107
SAS_-_Module_2_-_Business_Architecture_v4.3.pdf                                                80
SAS_-_Module_3.1_-_ASRs_v4.1.pdf                                                               66
SAS_-_Module_3.2_-_Quality_Attributes_v4.4.pdf                                                 80
SAS_-_Module_4_-_Architectural_Styles_and_Patterns_v4.1.pdf                                   125
SAS_-_Module_5_-_Architecture_Modeling_v4.pdf                                                  75
SAS_-_Module_6_-_Architecture_Documentation_v4.pdf                                            138
SAS_-_Module_7.1_-_Pre-sales_v4.pdf                                                            85
SAS_-_Module_7.2_-_Estimation_v4.0.pdf                                                         61
SAS_-_Module_7.3_-_Discovery__Constructio

## Step 5 — Embeddings & ChromaDB Vector Store

Each chunk is converted to a dense vector with a sentence-transformer model and stored in ChromaDB.
The store is persisted to disk so you can reload it without re-embedding.

> **Tip:** If you change the PDF set or the embedding model, delete the `vectorstore/` folder first.

In [5]:

import chromadb
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# ── 1. Embedding model ────────────────────────────────────────────────────
print(f"Loading embedding model: {EMBEDDING_MODEL} …")
embedding = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print("Embedding model ready.\n")

# ── 2. Persist directory & collection name ────────────────────────────────
persist_dir     = str(VECTORSTORE_DIR.resolve())
collection_name = "rag_docs"
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

# ── 3. Build or reload the vector store ───────────────────────────────────
chroma_client = chromadb.PersistentClient(path=persist_dir)

# Check whether the collection already contains documents
existing   = chroma_client.list_collections()
has_data   = any(c.name == collection_name for c in existing)

if has_data:
    col           = chroma_client.get_collection(collection_name)
    collection_count = col.count()
    has_data      = collection_count > 0

if has_data:
    print(f"Re-using existing ChromaDB collection '{collection_name}' "
          f"({collection_count} vectors).")
    vectordb = Chroma(
        collection_name=collection_name,
        embedding_function=embedding,
        persist_directory=persist_dir,
        client=chroma_client,
    )
else:
    print(f"Building ChromaDB collection '{collection_name}' "
          f"from {len(chunked_docs)} chunks …")
    vectordb = Chroma.from_documents(
        documents=chunked_docs,
        embedding=embedding,
        collection_name=collection_name,
        persist_directory=persist_dir,
        client=chroma_client,
    )
    collection_count = vectordb._collection.count()
    print(f"Stored {collection_count} vectors.\n")

# ── 4. Retriever ──────────────────────────────────────────────────────────
retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": TOP_K_RETRIEVAL,
        "fetch_k": TOP_K_RETRIEVAL * 4,
        "lambda_mult": 0.6,
    },
)

print(f"\n✅ Vector store ready  — {collection_count} vectors in '{collection_name}'")
print(f"   Retriever          : MMR, top-{TOP_K_RETRIEVAL}")
print(f"   Persist directory  : {persist_dir}")


Loading embedding model: all-mpnet-base-v2 …


/var/folders/gb/ynrhs_m16_n2_mx7_yw3qrcc0000gn/T/ipykernel_66200/3283251809.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the langchain-huggingface package and should be used instead. To use it run `pip install -U langchain-huggingface` and import as `from langchain_huggingface import HuggingFaceEmbeddings`.
  embedding = HuggingFaceEmbeddings(
/Users/Dzmitry_Luhavy/Documents/SASAI/RAG/.venv/lib/python3.13/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange
Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
/var/folders/gb/ynrhs_m16_n2_mx7_yw3qrcc0000gn/T/ipykernel_66200/3283251809.py:34: LangChainDeprecationWar

Embedding model ready.

Re-using existing ChromaDB collection 'rag_docs' (1681 vectors).

✅ Vector store ready  — 1681 vectors in 'rag_docs'
   Retriever          : MMR, top-4
   Persist directory  : /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/vectorstore


## Step 6 — RAG Chain Construction

Builds a `ConversationalRetrievalChain` that:
1. Fetches the top-K most relevant chunks from ChromaDB for each question (MMR)
2. **Re-ranks them with the Cohere cross-encoder** (`CohereRerank`) when a Cohere API key is present — upgrades retrieval from diversity-only (MMR) to relevance-ranked
3. Passes the final ranked context to the LLM alongside conversation history
4. Returns a grounded answer with source documents

Supports both **OpenAI** and **Cohere** LLMs — controlled by `LLM_PROVIDER` in Step 2.
The reranker activates independently of the LLM choice, as long as `COHERE_API_KEY` is set.


In [6]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory
from langchain.prompts import PromptTemplate
from langchain.retrievers import ContextualCompressionRetriever

# ── LLM selection ──────────────────────────────────────────────────────────
if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(
        model=OPENAI_MODEL,
        openai_api_key=OPENAI_API_KEY,
        temperature=0,
    )
    print(f"LLM: OpenAI {OPENAI_MODEL}")

elif LLM_PROVIDER == "cohere":
    from langchain_cohere import ChatCohere
    llm = ChatCohere(
        model=COHERE_MODEL,
        cohere_api_key=COHERE_API_KEY,
        temperature=0,
    )
    print(f"LLM: Cohere {COHERE_MODEL}")

else:
    raise ValueError(f"Unknown LLM_PROVIDER: '{LLM_PROVIDER}'. Choose 'openai' or 'cohere'.")

# ── Cohere cross-encoder reranker (activates when COHERE_API_KEY is set) ──
# Wraps the MMR retriever: MMR fetches candidates, reranker scores by relevance.
reranker = None
active_retriever = retriever  # falls back to plain MMR if no reranker

if COHERE_API_KEY:
    try:
        from langchain_cohere import CohereRerank
        reranker = CohereRerank(cohere_api_key=COHERE_API_KEY, top_n=3)
        active_retriever = ContextualCompressionRetriever(
            base_compressor=reranker,
            base_retriever=retriever,
        )
        print("Reranker: CohereRerank (top_n=3) — MMR candidates re-ranked by relevance")
    except Exception as e:
        print(f"⚠️  Cohere reranker unavailable ({e}); falling back to plain MMR retriever")
else:
    print("Reranker: disabled (COHERE_API_KEY not set) — using plain MMR retriever")

# ── Custom QA prompt — allows synthesis, not just extraction ───────────────
QA_PROMPT = PromptTemplate(
    input_variables=["context", "question"],
    template=(
        "You are an expert assistant for a Software Solution Architecture programme.\n"
        "Use the retrieved document excerpts below to answer the question.\n"
        "Synthesise information across all excerpts when summarising.\n"
        "If the excerpts don't contain enough information, say so briefly but still "
        "share what you can infer from them.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
)

# ── Chain with conversational memory ──────────────────────────────────────
memory = ConversationBufferMemory(
    return_messages=True,
    memory_key="chat_history",
    output_key="answer",
)

qa_chain = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=active_retriever,
    memory=memory,
    return_source_documents=True,
    combine_docs_chain_kwargs={"prompt": QA_PROMPT},
    verbose=False,
)

print("RAG chain ready.\n")


LLM: OpenAI gpt-4o-mini
Reranker: CohereRerank (top_n=3) — MMR candidates re-ranked by relevance
RAG chain ready.



## Step 7 — Batch Document Analysis

For every unique PDF file, run the structured analysis queries defined in Step 2.
Each query uses a **document-scoped MMR retriever** — ChromaDB is queried with a
`filter={"filename": filename}` so only chunks from that specific file are retrieved.

The retrieved excerpts are assembled as context and passed to the LLM (true RAG path).
Each result row includes a `source_chunks` field listing the chunk IDs used — providing
source attribution on the batch analysis path, not just the interactive Q&A path.


In [8]:
import time
from langchain_core.prompts import ChatPromptTemplate
from langchain.chains import LLMChain
from langchain.retrievers import ContextualCompressionRetriever

# ── Rate-limit settings (Cohere trial key: 10 req/min) ───────────────────
# Set RERANKER_CALL_DELAY > 0 (e.g. 7) to proactively space out reranker calls.
# At 0 the retry-on-429 logic handles it reactively (faster when quota is fresh).
RERANKER_CALL_DELAY = 0   # seconds between reranker calls (0 = disabled)
_RETRY_WAIT_SECS    = 65  # seconds to wait after a 429 before retrying


def _invoke_with_retry(retriever, question: str, max_retries: int = 5) -> list:
    """
    Invoke a retriever with automatic exponential-backoff retry on
    Cohere 429 TooManyRequestsError (trial key: 10 calls / minute).
    """
    for attempt in range(max_retries):
        try:
            result = retriever.invoke(question)
            if RERANKER_CALL_DELAY > 0:
                time.sleep(RERANKER_CALL_DELAY)
            return result
        except Exception as exc:
            is_rate_limit = (
                type(exc).__name__ == "TooManyRequestsError"
                or getattr(exc, "status_code", None) == 429
            )
            if is_rate_limit and attempt < max_retries - 1:
                wait = _RETRY_WAIT_SECS * (attempt + 1)
                print(f"    ⏳ Rate limit hit — waiting {wait}s "
                      f"(retry {attempt + 1}/{max_retries - 1}) …")
                time.sleep(wait)
            else:
                raise


def analyse_document(filename: str, queries: dict) -> dict:
    """
    Run each structured query against ChromaDB using a document-scoped MMR retriever.
    Retrieves the top-K most relevant chunks for each query (filtered to this filename),
    optionally re-ranks them with the Cohere cross-encoder, assembles them as context,
    and calls the LLM to produce a grounded answer.
    Returns the results dict including source_chunks for traceability.
    """
    # Document-scoped retriever: MMR search restricted to chunks from this file only
    base_doc_retriever = vectordb.as_retriever(
        search_type="mmr",
        search_kwargs={
            "k": TOP_K_RETRIEVAL,
            "fetch_k": TOP_K_RETRIEVAL * 4,
            "lambda_mult": 0.6,
            "filter": {"filename": filename},
        },
    )

    # Wrap with Cohere reranker if available (same reranker instance from Step 6)
    doc_retriever = (
        ContextualCompressionRetriever(
            base_compressor=reranker,
            base_retriever=base_doc_retriever,
        )
        if reranker is not None
        else base_doc_retriever
    )

    results = {"filename": filename}
    all_source_chunk_ids: set[str] = set()

    for field, question in queries.items():
        # Retrieve relevant chunks scoped to this document (with rate-limit retry)
        retrieved_docs = _invoke_with_retry(doc_retriever, question)
        context = "\n\n---\n\n".join(doc.page_content for doc in retrieved_docs)
        chunk_ids = [doc.metadata.get("chunk_id", "?") for doc in retrieved_docs]
        all_source_chunk_ids.update(str(c) for c in chunk_ids)

        prompt = ChatPromptTemplate.from_template(
            "You are an expert document analyst.\n\n"
            "Relevant excerpts from the document:\n{context}\n\n"
            "Task: {question}\n\n"
            "Answer concisely and factually based solely on the excerpts above."
        )
        chain = LLMChain(llm=llm, prompt=prompt)
        answer = chain.run(context=context, question=question)
        results[field] = answer.strip()

    results["source_chunks"] = ", ".join(sorted(all_source_chunk_ids, key=lambda x: int(x) if x.isdigit() else x))
    return results


# ── Group raw pages by filename (kept for page/char stats) ────────────────
from collections import defaultdict

docs_by_file: dict[str, str] = defaultdict(str)
for doc in raw_docs:
    fname = doc.metadata.get("filename", "unknown")
    docs_by_file[fname] += doc.page_content + "\n"

# ── Unique filenames derived from indexed chunks ──────────────────────────
unique_files = sorted({doc.metadata.get("filename") for doc in chunked_docs})
print(f"Documents to analyse: {unique_files}\n")

# ── Run analysis ──────────────────────────────────────────────────────────
analysis_results = []

for fname in unique_files:
    print(f"Analysing: {fname} ...")
    row = analyse_document(fname, ANALYSIS_QUERIES)
    row["total_pages"]  = sum(1 for d in raw_docs   if d.metadata.get("filename") == fname)
    row["total_chunks"] = sum(1 for d in chunked_docs if d.metadata.get("filename") == fname)
    row["total_chars"]  = len(docs_by_file[fname])
    analysis_results.append(row)
    preview = row["source_chunks"][:60] + ("…" if len(row["source_chunks"]) > 60 else "")
    print(f"  ✓ {fname} done  (source chunks: {preview})")

print(f"\n✅ Analysis complete.  {len(analysis_results)} document(s) processed.")


Documents to analyse: ['SAS_-_Module_1_-_Program___SA_Introduction__eLearning__v4.3.pdf', 'SAS_-_Module_2_-_Business_Architecture_v4.3.pdf', 'SAS_-_Module_3.1_-_ASRs_v4.1.pdf', 'SAS_-_Module_3.2_-_Quality_Attributes_v4.4.pdf', 'SAS_-_Module_4_-_Architectural_Styles_and_Patterns_v4.1.pdf', 'SAS_-_Module_5_-_Architecture_Modeling_v4.pdf', 'SAS_-_Module_6_-_Architecture_Documentation_v4.pdf', 'SAS_-_Module_7.1_-_Pre-sales_v4.pdf', 'SAS_-_Module_7.2_-_Estimation_v4.0.pdf', 'SAS_-_Module_7.3_-_Discovery__Construction__Transition_v4.1.pdf', 'SAS_-_Module_8.1_-_Architecture_Design_v4.pdf', 'SAS_-_Module_8.2_-_Architecture_Review_v4.pdf', 'SAS_-_Module_9.1.1_-_Technology_Domains_-_Cloud.Virtualization___Containerization_v4.2.pdf', 'SAS_-_Module_9.1.2_-_Technology_Domains_-_Cloud.Core_v4.2.pdf', 'SAS_-_Module_9.2_-_Technology_Domains_-_Web_and_Mobile_v4.2.pdf', 'SAS_-_Module_9.3_-_Technology_Domains_-_Cache_v4.1.pdf', 'SAS_-_Module_9.4_-_Technology_Domains_-_Data__AI___ML_v4.pdf', 'SAS_-_Module

## Step 8 — Export Results to CSV

Flatten the analysis dictionary into a `pandas` DataFrame and write to `output/analysis_results.csv`.

In [9]:
import pandas as pd

# Build DataFrame — each row is one PDF document
results_df = pd.DataFrame(analysis_results)

# Reorder columns for readability; source_chunks at the end for traceability
ordered_cols = (
    ["filename", "total_pages", "total_chunks", "total_chars"]
    + list(ANALYSIS_QUERIES.keys())
    + ["source_chunks"]
)
results_df = results_df[[c for c in ordered_cols if c in results_df.columns]]

# Save to CSV (UTF-8 with BOM for Excel compatibility)
results_df.to_csv(CSV_OUTPUT, index=False, encoding="utf-8-sig")

print(f"CSV saved → {CSV_OUTPUT.resolve()}")
print(f"Rows: {len(results_df)}  |  Columns: {list(results_df.columns)}\n")

# Preview
results_df.head()


CSV saved → /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/output/analysis_results.csv
Rows: 20  |  Columns: ['filename', 'total_pages', 'total_chunks', 'total_chars', 'summary', 'learning_objectives', 'module_topic', 'architecture_patterns', 'technologies_tools', 'architectural_decisions', 'difficulty_level', 'prerequisites', 'exercises_labs', 'key_concepts', 'gaps_recommendations', 'source_chunks']



,filename,total_pages,total_chunks,total_chars,summary,learning_objectives,module_topic,architecture_patterns,technologies_tools,architectural_decisions,difficulty_level,prerequisites,exercises_labs,key_concepts,gaps_recommendations,source_chunks
0,SAS_-_Module_1_-_Program___SA_Introduction__eL...,86,107,46704,The document serves as a resource guide for se...,1. Understand the role of solution architects ...,Solution Architecture,The excerpts provided do not explicitly mentio...,The excerpts do not explicitly mention any spe...,The excerpts focus on the architecture design ...,Introductory,None stated,"Yes, there are practical exercises and assignm...","Conceptual Architecture, Logical Architecture,...","Based on the provided excerpts, several potent...","4, 7, 8, 19, 20, 22, 26, 30, 33, 37, 41, 50, 5..."
1,SAS_-_Module_2_-_Business_Architecture_v4.3.pdf,63,80,39181,The document emphasizes the importance of unde...,1. Understanding the importance of resource al...,Enterprise Architecture,The excerpts provided do not mention any speci...,The excerpts provided do not reference any spe...,The document discusses the importance of manag...,Intermediate,None stated,None.,"Interviews, Item Tracking, Lessons Learned, Me...",1. **Lack of Depth on Business Processes**: Th...,"107, 121, 123, 126, 130, 138, 140, 143, 157, 1..."
2,SAS_-_Module_3.1_-_ASRs_v4.1.pdf,61,66,22028,The document outlines key references and metho...,1. Understand the importance of liability prot...,Solution Architecture,The excerpts provided do not explicitly mentio...,CORBA,The document discusses the prioritization of a...,Intermediate,None stated,"Yes, there are practical exercises described. ...","Architecturally Significant Requirements, Func...","Based on the provided excerpts, the following ...","188, 189, 190, 198, 206, 210, 213, 214, 222, 2..."
3,SAS_-_Module_3.2_-_Quality_Attributes_v4.4.pdf,63,80,46267,The document outlines a module focused on Qual...,1. Understand the importance of security as a ...,Solution Architecture,The excerpts provided do not explicitly mentio...,ZAP Proxy,The excerpts highlight the importance of consi...,Intermediate,None stated,None.,"Architecture, System Design, Quality Attribute...","Based on the provided excerpts, the following ...","254, 255, 257, 263, 274, 288, 296, 314, 315, 3..."
4,SAS_-_Module_4_-_Architectural_Styles_and_Patt...,104,125,60709,The document discusses the design principle of...,1. Understand the concept of anarchic scalabil...,Microservices,The excerpts provided do not explicitly mentio...,"Azure, Microsoft Power BI",The document discusses the significant impact ...,Intermediate,None stated,"Yes, the document describes a practical exerci...","Architectural Style, Architectural Pattern, Se...","Based on the provided excerpts, here are some ...","333, 337, 338, 340, 342, 358, 363, 372, 374, 3..."


## Step 9 — Visualisations with Plotly

Six charts + one relationship diagram, tailored to the Software Solution Architecture curriculum:

**Charts:**
1. **Document Size Comparison** — pages & chunks per PDF
2. **Difficulty Level Distribution** — Introductory / Intermediate / Advanced breakdown
3. **Module Topics** — documents per topic (horizontal bar)
4. **Top Architecture Patterns** — frequency across all documents
5. **Top Technologies & Tools** — frequency across all documents
6. **Document Similarity Heatmap** — cosine similarity between document embeddings

**Relationship Diagram:**
7. **Topic → Technology Sankey** — flow diagram linking each module topic to the technologies it references, showing cross-topic technology sharing


In [10]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display
import numpy as np
import pandas as pd

# Ensure plotly renders correctly inside Jupyter
import plotly.io as pio
pio.renderers.default = "notebook_connected"

# ── Load results: prefer CSV on disk, fall back to in-memory DataFrame ─────
if CSV_OUTPUT.exists():
    results_df = pd.read_csv(CSV_OUTPUT, encoding="utf-8-sig")
    print(f"✅ Loaded results from CSV: {CSV_OUTPUT.resolve()}  ({len(results_df)} row(s))")
elif "results_df" in dir() and results_df is not None:
    print("⚠️  CSV not found — using in-memory DataFrame from Step 8.")
else:
    raise RuntimeError(
        f"No data available. Run Step 7–8 first, or ensure '{CSV_OUTPUT}' exists."
    )

print(f"   Columns: {list(results_df.columns)}\n")

# ── Helper: split comma-separated column into a frequency Series ────────────
def explode_csv_col(df: pd.DataFrame, col: str, top_n: int = 20) -> pd.DataFrame:
    items = []
    for cell in df[col].dropna():
        items.extend([v.strip() for v in str(cell).split(",") if v.strip()])
    freq = pd.Series(items).value_counts().head(top_n).reset_index()
    freq.columns = [col, "Frequency"]
    return freq


# ── Chart 1: Document Size (pages + chunks) ─────────────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=["Pages per Document", "Chunks per Document"]
)
fig1.add_trace(
    go.Bar(x=results_df["filename"], y=results_df["total_pages"],
           name="Pages", marker_color="#4C78A8"),
    row=1, col=1
)
fig1.add_trace(
    go.Bar(x=results_df["filename"], y=results_df["total_chunks"],
           name="Chunks", marker_color="#72B7B2"),
    row=1, col=2
)
fig1.update_layout(title_text="Document Size Comparison", showlegend=False, height=420)
display(fig1)


# ── Chart 2: Difficulty Level Distribution ──────────────────────────────────
if "difficulty_level" in results_df.columns:
    diff_counts = (
        results_df["difficulty_level"]
        .str.strip()
        .str.capitalize()
        .value_counts()
        .reindex(["Introductory", "Intermediate", "Advanced"], fill_value=0)
        .reset_index()
    )
    diff_counts.columns = ["Level", "Count"]

    colour_map = {"Introductory": "#2ECC71", "Intermediate": "#F39C12", "Advanced": "#E74C3C"}
    fig2 = px.bar(
        diff_counts,
        x="Level", y="Count",
        color="Level",
        color_discrete_map=colour_map,
        title="Difficulty Level Distribution",
        text="Count",
    )
    fig2.update_traces(textposition="outside")
    fig2.update_layout(height=400, showlegend=False)
    display(fig2)


# ── Chart 3: Module Topics — Documents per Topic ───────────────────────────
if "module_topic" in results_df.columns:
    topics_df = (
        results_df[["filename", "module_topic"]]
        .dropna()
        .assign(module_topic=lambda d: d["module_topic"].str.strip())
    )

    # Aggregate: count documents per topic, collect filenames for hover
    topic_agg = (
        topics_df
        .groupby("module_topic", sort=False)
        .agg(
            count=("filename", "count"),
            documents=("filename", lambda x: "<br>".join(x.tolist())),
        )
        .reset_index()
        .sort_values("count", ascending=True)
    )

    fig3 = px.bar(
        topic_agg,
        x="count",
        y="module_topic",
        orientation="h",
        title="Documents per Module Topic",
        text="count",
        color="count",
        color_continuous_scale="Blues",
        labels={"module_topic": "Topic", "count": "# Documents"},
        custom_data=["documents"],
    )
    fig3.update_traces(
        textposition="outside",
        hovertemplate="<b>%{y}</b><br>Documents: %{x}<br><br>%{customdata[0]}<extra></extra>",
    )
    fig3.update_layout(height=max(400, len(topic_agg) * 45), showlegend=False,
                       coloraxis_showscale=False)
    display(fig3)


# ── Chart 4: Top Architecture Patterns ──────────────────────────────────────
if "architecture_patterns" in results_df.columns:
    patterns_freq = explode_csv_col(results_df, "architecture_patterns", top_n=20)
    fig4 = px.bar(
        patterns_freq,
        x="Frequency", y="architecture_patterns",
        orientation="h",
        title="Top Architecture Patterns Across All Documents",
        color="Frequency",
        color_continuous_scale="Blues",
        labels={"architecture_patterns": "Pattern"},
    )
    fig4.update_layout(height=520, yaxis={"autorange": "reversed"})
    display(fig4)


# ── Chart 5: Top Technologies & Tools ───────────────────────────────────────
if "technologies_tools" in results_df.columns:
    tech_freq = explode_csv_col(results_df, "technologies_tools", top_n=20)
    fig5 = px.bar(
        tech_freq,
        x="Frequency", y="technologies_tools",
        orientation="h",
        title="Top Technologies & Tools Across All Documents",
        color="Frequency",
        color_continuous_scale="Teal",
        labels={"technologies_tools": "Technology / Tool"},
    )
    fig5.update_layout(height=520, yaxis={"autorange": "reversed"})
    display(fig5)


# ── Chart 6: Document Embedding Similarity Heatmap ──────────────────────────
if "docs_by_file" in dir() and len(docs_by_file) > 1:
    doc_names  = list(docs_by_file.keys())
    doc_texts  = [docs_by_file[n][:4000] for n in doc_names]

    doc_vectors = np.array(embedding.embed_documents(doc_texts))
    norms       = np.linalg.norm(doc_vectors, axis=1, keepdims=True)
    sim_matrix  = (doc_vectors @ doc_vectors.T) / (norms @ norms.T + 1e-9)

    fig6 = px.imshow(
        sim_matrix,
        x=doc_names, y=doc_names,
        text_auto=".2f",
        color_continuous_scale="RdBu",
        title="Document Similarity Heatmap (Cosine)",
        zmin=0, zmax=1,
    )
    fig6.update_layout(height=520)
    display(fig6)
else:
    print("ℹ️  Similarity heatmap skipped — requires ≥ 2 documents and docs_by_file in memory.")


# ── Diagram 7: Topic → Technology Sankey (relationship diagram) ─────────────
# Nodes: module topics (left) → top-15 technologies (right)
# Edges: document links a topic to each technology it references
fig7 = None
if "module_topic" in results_df.columns and "technologies_tools" in results_df.columns:
    edge_rows = []
    for _, row in results_df.dropna(subset=["module_topic", "technologies_tools"]).iterrows():
        topic = str(row["module_topic"]).strip()
        for tech in str(row["technologies_tools"]).split(","):
            tech = tech.strip()
            if tech:
                edge_rows.append({"topic": topic, "tech": tech})

    if edge_rows:
        edges_df  = pd.DataFrame(edge_rows)
        top_techs = edges_df["tech"].value_counts().head(15).index.tolist()
        edges_df  = edges_df[edges_df["tech"].isin(top_techs)]
        flow_df   = edges_df.groupby(["topic", "tech"]).size().reset_index(name="value")

        topics    = sorted(flow_df["topic"].unique().tolist())
        techs     = sorted(flow_df["tech"].unique().tolist())
        all_nodes = topics + techs
        node_idx  = {n: i for i, n in enumerate(all_nodes)}

        node_colors = (
            ["rgba(44, 95, 158, 0.85)"] * len(topics)    # blue for topics
            + ["rgba(72, 183, 178, 0.85)"] * len(techs)  # teal for technologies
        )

        fig7 = go.Figure(go.Sankey(
            arrangement="snap",
            node=dict(
                pad=18,
                thickness=20,
                line=dict(color="#aaa", width=0.5),
                label=all_nodes,
                color=node_colors,
            ),
            link=dict(
                source=[node_idx[r["topic"]] for _, r in flow_df.iterrows()],
                target=[node_idx[r["tech"]]  for _, r in flow_df.iterrows()],
                value=flow_df["value"].tolist(),
                color="rgba(180, 200, 220, 0.4)",
            ),
        ))
        fig7.update_layout(
            title_text="Topic → Technology Relationship (Sankey Diagram)",
            font_size=12,
            height=620,
        )
        display(fig7)
    else:
        print("ℹ️  Sankey skipped — no topic/technology edge data found.")
else:
    print("ℹ️  Sankey skipped — requires module_topic and technologies_tools columns.")

print("\n✅ All visualisations rendered.")


✅ Loaded results from CSV: /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/output/analysis_results.csv  (20 row(s))
   Columns: ['filename', 'total_pages', 'total_chunks', 'total_chars', 'summary', 'learning_objectives', 'module_topic', 'architecture_patterns', 'technologies_tools', 'architectural_decisions', 'difficulty_level', 'prerequisites', 'exercises_labs', 'key_concepts', 'gaps_recommendations', 'source_chunks']




✅ All visualisations rendered.


## Step 10 — RAG Evaluation & Quality Metrics

Measures retrieval and generation quality against a small **golden dataset** of
5 representative Software Architecture questions.  No external evaluation API required.

Two lightweight RAGAS-style metrics are computed locally:

| Metric | What it measures |
|---|---|
| **Context Recall** | Fraction of reference-answer key tokens found in the retrieved chunks |
| **Answer Faithfulness** | LLM-verified: are all factual claims in the answer supported by the context? |

A summary table with per-question scores and corpus-level means is printed at the end.


In [11]:
import re
import pandas as pd
from IPython.display import display

# ── Golden dataset ─────────────────────────────────────────────────────────
# 5 representative questions with reference answers.
# Reference answers express the *key concepts* expected — not exact LLM wording.
GOLDEN_DATASET = [
    {
        "question": "What are the key differences between monolithic and microservices architectural styles, and what are their trade-offs?",
        "reference": (
            "Monolith deploys as a single unit with tight coupling; microservices decompose "
            "the system into independently deployable services with loose coupling and high cohesion, "
            "enabling independent scaling but adding distributed system complexity."
        ),
    },
    {
        "question": "What is FURPS+ and how is it used to classify quality attributes?",
        "reference": (
            "FURPS+ categorises quality attributes into Functionality, Usability, Reliability, "
            "Performance, and Supportability, plus additional categories such as constraints and "
            "design requirements. It is used to systematically identify and document non-functional requirements."
        ),
    },
    {
        "question": "What is ATAM and what is the role of a Utility Tree in architecture review?",
        "reference": (
            "ATAM (Architecture Tradeoffs Analysis Method) is a structured method for evaluating "
            "architectural approaches against quality attribute requirements. A Utility Tree organises "
            "ASRs by quality attribute, scenario, and priority to expose trade-offs and risks."
        ),
    },
    {
        "question": "What is the CAP theorem and how does it influence NoSQL database design?",
        "reference": (
            "The CAP theorem states a distributed system can guarantee only two of Consistency, "
            "Availability, and Partition tolerance simultaneously. It drives NoSQL design decisions "
            "around sharding, replication, and the trade-off between ACID compliance and horizontal scaling."
        ),
    },
    {
        "question": "What is Message-Oriented Middleware and what architectural role does a message broker play?",
        "reference": (
            "Message-Oriented Middleware (MOM) decouples producers and consumers using asynchronous "
            "message passing. A message broker routes messages between queues and exchanges using "
            "routing keys, enabling reliable delivery, load levelling, and event-driven integration."
        ),
    },
]


# ── Metric helpers ─────────────────────────────────────────────────────────

def tokenise(text: str) -> set[str]:
    """Lowercase word tokens, stopwords removed."""
    STOPWORDS = {"a","an","the","is","are","was","were","and","or","of","in",
                 "to","for","with","on","by","as","it","its","that","this",
                 "be","at","from","which","not","also","what","how","why"}
    return {w for w in re.findall(r"[a-z]+", text.lower()) if w not in STOPWORDS}


def context_recall(reference: str, retrieved_docs: list) -> float:
    """Fraction of reference key-tokens found in any retrieved chunk."""
    ref_tokens  = tokenise(reference)
    if not ref_tokens:
        return 0.0
    ctx_text    = " ".join(doc.page_content for doc in retrieved_docs)
    ctx_tokens  = tokenise(ctx_text)
    hit         = ref_tokens & ctx_tokens
    return round(len(hit) / len(ref_tokens), 3)


def answer_faithfulness(question: str, answer: str, retrieved_docs: list) -> float:
    """
    Ask the LLM: 'Is this answer fully supported by the provided context?'
    Returns 1.0 (yes) or 0.0 (no/partial).
    """
    context = "\n\n---\n\n".join(doc.page_content for doc in retrieved_docs)
    verdict_prompt = (
        f"Context:\n{context}\n\n"
        f"Answer: {answer}\n\n"
        "Is every factual claim in the answer directly supported by the context above?\n"
        "Reply with a single word: YES or NO."
    )
    try:
        response = llm.invoke(verdict_prompt)
        text = response.content if hasattr(response, "content") else str(response)
        return 1.0 if text.strip().lower().startswith("yes") else 0.0
    except Exception:
        return 0.0


# ── Run evaluation ─────────────────────────────────────────────────────────
# Clear conversation memory so evaluation runs from a clean state
memory.clear()
eval_rows = []

print("Running RAG evaluation on golden dataset …\n")
for item in GOLDEN_DATASET:
    q   = item["question"]
    ref = item["reference"]

    # Retrieve using the active_retriever (MMR + optional reranker, unfiltered)
    retrieved = active_retriever.invoke(q)
    ctx_len   = sum(len(d.page_content) for d in retrieved)

    # Generate answer via the RAG chain (let memory manage history)
    result    = qa_chain({"question": q})
    answer    = result["answer"]

    recall      = context_recall(ref, retrieved)
    faithfulness = answer_faithfulness(q, answer, retrieved)

    eval_rows.append({
        "Question":           q[:60] + ("…" if len(q) > 60 else ""),
        "Context chars":      ctx_len,
        "Context Recall":     recall,
        "Answer Faithfulness": faithfulness,
    })
    print(f"  Q: {q[:55]}…")
    print(f"     Recall={recall:.2f}  Faithfulness={faithfulness:.1f}  "
          f"Context={ctx_len} chars\n")

eval_df = pd.DataFrame(eval_rows)

# Mean row
means = eval_df[["Context Recall", "Answer Faithfulness"]].mean().round(3)
mean_row = {
    "Question": "── MEAN ──",
    "Context chars": int(eval_df["Context chars"].mean()),
    "Context Recall": means["Context Recall"],
    "Answer Faithfulness": means["Answer Faithfulness"],
}
eval_df = pd.concat([eval_df, pd.DataFrame([mean_row])], ignore_index=True)

print("=" * 60)
print("RAG Evaluation Summary")
print("=" * 60)
display(eval_df.style.format({
    "Context Recall": "{:.2f}",
    "Answer Faithfulness": "{:.1f}",
    "Context chars": "{:,}",
}).highlight_min(subset=["Context Recall", "Answer Faithfulness"], color="#ffe0e0")
  .highlight_max(subset=["Context Recall", "Answer Faithfulness"], color="#d4edda"))

print(f"\nMean Context Recall     : {means['Context Recall']:.2f}")

print(f"Mean Answer Faithfulness: {means['Answer Faithfulness']:.1f}")

Running RAG evaluation on golden dataset …



/var/folders/gb/ynrhs_m16_n2_mx7_yw3qrcc0000gn/T/ipykernel_66200/3980383599.py:108: LangChainDeprecationWarning:

The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use invoke instead.



  Q: What are the key differences between monolithic and mic…
     Recall=0.39  Faithfulness=1.0  Context=2106 chars

  Q: What is FURPS+ and how is it used to classify quality a…
     Recall=0.52  Faithfulness=1.0  Context=1244 chars

  Q: What is ATAM and what is the role of a Utility Tree in …
     Recall=0.61  Faithfulness=1.0  Context=854 chars

  Q: What is the CAP theorem and how does it influence NoSQL…
     Recall=0.54  Faithfulness=1.0  Context=2464 chars

  Q: What is Message-Oriented Middleware and what architectu…
     Recall=0.39  Faithfulness=1.0  Context=1643 chars

RAG Evaluation Summary


,Question,Context chars,Context Recall,Answer Faithfulness
0,What are the key differences between monolithic and microser…,"2,106",0.39,1.0
1,What is FURPS+ and how is it used to classify quality attrib…,"1,244",0.52,1.0
2,What is ATAM and what is the role of a Utility Tree in archi…,854,0.61,1.0
3,What is the CAP theorem and how does it influence NoSQL data…,"2,464",0.54,1.0
4,What is Message-Oriented Middleware and what architectural r…,"1,643",0.39,1.0
5,── MEAN ──,"1,662",0.49,1.0



Mean Context Recall     : 0.49
Mean Answer Faithfulness: 1.0


## Step 11 — HTML Report Generation

Combine the analysis table and all Plotly charts into a single, self-contained `report.html`
using a Jinja2 template.  The file can be opened directly in any browser — no server needed.

Sections in the report:
- Header with generation timestamp
- Executive Summary table (from CSV)
- Per-document detail cards (summary, topics, entities, recommendations)
- All interactive Plotly charts embedded as inline JavaScript


In [12]:
from jinja2 import Template
from datetime import datetime
import plotly.io as pio

# ── Serialise charts to self-contained HTML fragments ─────────────────────
def fig_to_html(fig, div_id: str) -> str:
    return pio.to_html(fig, full_html=False, div_id=div_id, include_plotlyjs="cdn")

charts_html = {}
charts_html["size_chart"]       = fig_to_html(fig1, "chart_size")
if "difficulty_level" in results_df.columns:
    charts_html["difficulty_chart"] = fig_to_html(fig2, "chart_difficulty")
if "module_topic" in results_df.columns:
    charts_html["topics_chart"]     = fig_to_html(fig3, "chart_topics")
if "architecture_patterns" in results_df.columns:
    charts_html["patterns_chart"]   = fig_to_html(fig4, "chart_patterns")
if "technologies_tools" in results_df.columns:
    charts_html["tech_chart"]       = fig_to_html(fig5, "chart_tech")
if "docs_by_file" in dir() and len(docs_by_file) > 1:
    charts_html["similarity_chart"] = fig_to_html(fig6, "chart_similarity")
if "fig7" in dir() and fig7 is not None:
    charts_html["sankey_chart"]     = fig_to_html(fig7, "chart_sankey")

# ── Jinja2 HTML template ───────────────────────────────────────────────────
HTML_TEMPLATE = """
<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>Software Solution Architecture — Learning Materials Analysis</title>
  <style>
    body      { font-family: 'Segoe UI', Arial, sans-serif; margin: 0; background: #f5f7fa; color: #333; }
    header    { background: #1a3a5c; color: white; padding: 24px 40px; }
    header h1 { margin: 0; font-size: 1.8em; }
    header p  { margin: 4px 0 0; opacity: 0.75; font-size: 0.9em; }
    main      { max-width: 1280px; margin: 30px auto; padding: 0 24px; }
    section   { background: white; border-radius: 8px; padding: 24px 28px; margin-bottom: 24px;
                box-shadow: 0 2px 8px rgba(0,0,0,0.08); }
    h2        { color: #1a3a5c; border-bottom: 2px solid #e3eaf3; padding-bottom: 8px; }
    h3        { color: #2c5f9e; margin-top: 20px; }
    table     { width: 100%; border-collapse: collapse; font-size: 0.85em; }
    th        { background: #1a3a5c; color: white; padding: 10px 12px; text-align: left; }
    td        { padding: 8px 12px; border-bottom: 1px solid #e8edf3; vertical-align: top; }
    tr:hover td { background: #f0f5ff; }
    .doc-card { border-left: 4px solid #2c5f9e; padding-left: 16px; margin-bottom: 28px; }
    .label    { font-weight: 600; color: #555; min-width: 200px; display: inline-block; vertical-align: top; }
    .tag      { display: inline-block; background: #e3eaf3; border-radius: 12px;
                padding: 2px 10px; margin: 2px 2px 4px; font-size: 0.8em; }
    .tag.pattern  { background: #dce8f5; color: #1a3a5c; }
    .tag.tech     { background: #d5f0e8; color: #1a5c3a; }
    .tag.concept  { background: #f5e8d5; color: #5c3a1a; }
    .badge        { display: inline-block; border-radius: 4px; padding: 2px 10px;
                    font-size: 0.8em; font-weight: 600; color: white; }
    .badge.Introductory { background: #2ECC71; }
    .badge.Intermediate { background: #F39C12; }
    .badge.Advanced     { background: #E74C3C; }
    footer    { text-align: center; padding: 20px; color: #888; font-size: 0.8em; }
  </style>
</head>
<body>
<header>
  <h1>Software Solution Architecture — Learning Materials Analysis</h1>
  <p>Generated: {{ generated_at }} &nbsp;|&nbsp; Documents: {{ doc_count }} &nbsp;|&nbsp; Model: {{ llm_info }}</p>
</header>
<main>

  <section>
    <h2>Executive Summary</h2>
    {{ summary_table }}
  </section>

  <section>
    <h2>Document Details</h2>
    {% for row in rows %}
    <div class="doc-card">
      <h3>{{ row.filename }}</h3>
      <p>
        <span class="label">Topic:</span> {{ row.get('module_topic', '—') }}
        &nbsp;&nbsp;
        <span class="label">Difficulty:</span>
        <span class="badge {{ row.get('difficulty_level','').strip().capitalize() }}">
          {{ row.get('difficulty_level', '—') }}
        </span>
        &nbsp;&nbsp;
        <span class="label">Pages / Chunks:</span> {{ row.get('total_pages','?') }} / {{ row.get('total_chunks','?') }}
      </p>
      <p><span class="label">Summary:</span> {{ row.get('summary', '—') }}</p>
      <p><span class="label">Learning Objectives:</span> {{ row.get('learning_objectives', '—') }}</p>
      <p><span class="label">Architecture Patterns:</span><br>
        {% for t in row.get('architecture_patterns','').split(',') %}
          {% if t.strip() %}<span class="tag pattern">{{ t.strip() }}</span>{% endif %}
        {% endfor %}
      </p>
      <p><span class="label">Technologies & Tools:</span><br>
        {% for t in row.get('technologies_tools','').split(',') %}
          {% if t.strip() %}<span class="tag tech">{{ t.strip() }}</span>{% endif %}
        {% endfor %}
      </p>
      <p><span class="label">Key Concepts:</span><br>
        {% for t in row.get('key_concepts','').split(',') %}
          {% if t.strip() %}<span class="tag concept">{{ t.strip() }}</span>{% endif %}
        {% endfor %}
      </p>
      <p><span class="label">Architectural Decisions:</span> {{ row.get('architectural_decisions', '—') }}</p>
      <p><span class="label">Prerequisites:</span> {{ row.get('prerequisites', '—') }}</p>
      <p><span class="label">Exercises / Labs:</span> {{ row.get('exercises_labs', '—') }}</p>
      <p><span class="label">Gaps & Recommendations:</span> {{ row.get('gaps_recommendations', '—') }}</p>
    </div>
    {% endfor %}
  </section>

  <section>
    <h2>Charts</h2>
    {{ charts_html.get('size_chart', '') }}
    {{ charts_html.get('difficulty_chart', '') }}
    {{ charts_html.get('topics_chart', '') }}
    {{ charts_html.get('patterns_chart', '') }}
    {{ charts_html.get('tech_chart', '') }}
    {{ charts_html.get('similarity_chart', '') }}
  </section>

  {% if charts_html.get('sankey_chart') %}
  <section>
    <h2>Relationship Diagram</h2>
    <p style="color:#555; font-size:0.9em;">
      Sankey flow diagram: each module topic (left, blue) is connected to the technologies
      it references (right, teal). Link width represents document frequency.
    </p>
    {{ charts_html.get('sankey_chart', '') }}
  </section>
  {% endif %}

</main>
<footer>RAG Pipeline • LangChain + ChromaDB + Plotly &nbsp;|&nbsp; Software Solution Architecture Program</footer>
</body>
</html>
"""

# ── Render with data ───────────────────────────────────────────────────────
llm_info = f"{LLM_PROVIDER} / {OPENAI_MODEL if LLM_PROVIDER == 'openai' else COHERE_MODEL}"

summary_cols = ["filename", "module_topic", "difficulty_level", "total_pages", "total_chunks"]
summary_cols = [c for c in summary_cols if c in results_df.columns]

summary_html = results_df[summary_cols].to_html(
    index=False, escape=True, border=0, classes="",
)

template    = Template(HTML_TEMPLATE)
html_output = template.render(
    generated_at  = datetime.now().strftime("%Y-%m-%d %H:%M"),
    doc_count     = len(results_df),
    llm_info      = llm_info,
    summary_table = summary_html,
    rows          = results_df.to_dict(orient="records"),
    charts_html   = charts_html,
)

HTML_REPORT.write_text(html_output, encoding="utf-8")
print(f"HTML report saved → {HTML_REPORT.resolve()}")

from IPython.display import IFrame
IFrame(src=str(HTML_REPORT.resolve()), width="100%", height=650)


HTML report saved → /Users/Dzmitry_Luhavy/Documents/SASAI/RAG/output/report.html


## Bonus — Interactive Q&A

Ask any question across all loaded PDFs.  The chain uses conversation memory,
so follow-up questions work naturally.

In [15]:
def ask(question: str, show_sources: bool = True) -> str:
    """
    Query the RAG chain and display the answer + source document snippets.
    Conversation history is preserved across calls within this session via
    the chain's ConversationBufferMemory — do NOT pass chat_history manually.
    """
    result = qa_chain({"question": question})
    answer = result["answer"]
    
    print(f"\n💬 Q: {question}")
    print(f"\n🤖 A: {answer}")
    
    if show_sources and result.get("source_documents"):
        print("\n📄 Sources:")
        for i, doc in enumerate(result["source_documents"], 1):
            fname = doc.metadata.get("filename", "unknown")
            page  = doc.metadata.get("page", "?")
            print(f"  [{i}] {fname}  (page {page})")
            print(f"      {doc.page_content[:200].strip()}…")
    
    return answer


# ── Reset memory so interactive Q&A starts from a clean conversation ───
memory.clear()

# ── Change the question below and re-run this cell ─────────────────────
# USER_QUESTION = "What architectural topics and key concepts are covered across these documents?"
USER_QUESTION = "What is the CAP theorem and how does it influence NoSQL database design?"


answer = ask(USER_QUESTION)


💬 Q: What is the CAP theorem and how does it influence NoSQL database design?

🤖 A: The CAP theorem, which stands for Consistency, Availability, and Partition Tolerance, posits that in a distributed data store, it is impossible to simultaneously guarantee all three of these properties. Instead, a system can only provide two out of the three at any given time. This theorem significantly influences NoSQL database design, as many NoSQL systems prioritize availability and partition tolerance over strict consistency, leading to the adoption of the BASE (Basically Available, Soft state, Eventually consistent) model instead of the traditional ACID (Atomicity, Consistency, Isolation, Durability) properties.

In a BASE architecture, databases are designed to be highly available, meaning they will respond to requests even if the data is not in a consistent state. This approach allows for greater flexibility in scaling, as it accommodates the eventual consistency of data rather than requiring im